# Historical Data EDA

Notebook dedicado a carga, limpeza, transforma??o `log_diff` e an?lise dos targets multiclasses.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.analysis.blockchain_eda import build_missing_report, dataset_snapshot
from src.analysis.historical_eda import analyze_targets, analyze_threshold_effects, compare_series_snapshots, target_correlation_matrix
from src.data.historical_loader import load_historical_asset_dataframe
from src.data.historical_preprocessing import add_price_direction_targets, prepare_historical_market_dataframe
from src.features.historical_features import DEFAULT_HISTORICAL_FEATURES, build_log_diff_dataset, create_multi_period_targets


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
PERIODS = (1, 3, 7, 14, 30)
THRESHOLD_MULTIPLIER = 0.75
FEATURE_COLUMNS = DEFAULT_HISTORICAL_FEATURES


In [ ]:
raw_btc_hist = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
raw_btc_hist.head()


In [ ]:
btc_hist = prepare_historical_market_dataframe(raw_btc_hist)
btc_hist = add_price_direction_targets(btc_hist, price_column='Close', horizons=(1,))
btc_hist.head()


In [ ]:
btc_hist_transformed = build_log_diff_dataset(btc_hist, feature_columns=FEATURE_COLUMNS, keep_close_column=True)
df_with_targets = create_multi_period_targets(
    btc_hist_transformed,
    periods=PERIODS,
    threshold_multiplier=THRESHOLD_MULTIPLIER,
)

df_with_targets.head()


In [ ]:
compare_series_snapshots(btc_hist, df_with_targets)


In [ ]:
dataset_snapshot(df_with_targets)


In [ ]:
build_missing_report(df_with_targets).head(20)


In [ ]:
target_analysis = analyze_targets(df_with_targets, periods=PERIODS)
target_analysis


In [ ]:
target_correlation_matrix(df_with_targets, periods=PERIODS)


In [ ]:
threshold_analysis = analyze_threshold_effects(
    btc_hist_transformed,
    target_builder=create_multi_period_targets,
    periods=PERIODS,
)
threshold_analysis.head()


In [ ]:
fig, axes = plt.subplots(len(PERIODS), 1, figsize=(12, 4 * len(PERIODS)))
for idx, period in enumerate(PERIODS):
    period_df = threshold_analysis[threshold_analysis['period'] == period]
    ax = axes[idx]
    ax.bar(period_df['threshold'], period_df['sell_ratio'], label='Sell (0)', alpha=0.7)
    ax.bar(period_df['threshold'], period_df['hold_ratio'], bottom=period_df['sell_ratio'], label='Hold (1)', alpha=0.7)
    ax.bar(
        period_df['threshold'],
        period_df['buy_ratio'],
        bottom=period_df['sell_ratio'] + period_df['hold_ratio'],
        label='Buy (2)',
        alpha=0.7,
    )
    ax.set_title(f'Period {period}d')
    ax.set_xlabel('Threshold multiplier')
    ax.set_ylabel('Ratio')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
